# Oscar DB — SQL Tutorial

Date: Created 2026-07-22 · Last updated 2026-07-22

Author: Wei He (Francis) · whorwhey@gmail.com

___

## Overview

A practical guide to SQL, taught through real questions against
`oscars.db` — a curated SQLite database of all Academy Awards
nominations and winners, 1st ceremony (1927/28) through the 98th (2025).

Each part introduces one syntax family, then asks a specific question
that puts it to work. Organized roughly simple → advanced, so earlier
parts build vocabulary that later parts rely on — but each section is
self-contained enough to jump back to as a reference.

### How to use this notebook

- **Learning path:** read straight through, Parts 0–8. Each query builds
  on syntax introduced earlier.
- **Reference lookup:** jump to the Contents below, find the syntax you
  need, read that section's notes.
- **Scratchpad:** copy a cell, change the parameters or filters,
  experiment. The `q()` helper in Part 0 runs any SQL you hand it.

### Contents

- **Part 0** — Setup (connection, `q()` helper)
- **Part 1** — Reading from one table (`SELECT`, `ORDER BY`, `DISTINCT`)
- **Part 2** — Filtering rows (`WHERE`, `LIKE`, parameters, `IS NULL`)
- **Part 3** — Combining tables (`JOIN`, chained joins, `LEFT JOIN`)
- **Part 4** — Aggregating (`GROUP BY`, `COUNT`, `SUM`, `HAVING`,
  `COUNT(DISTINCT)`, `GROUP_CONCAT`, `MIN`/`MAX`)
- **Part 5** — Set membership (`IN`, dynamic placeholders)
- **Part 6** — Subqueries (`NOT IN`, scalar subqueries, `OR`/`AND`
  mixing)
- **Part 7** — Alternative patterns (`NOT EXISTS`, CTEs)
- **Part 8** — Computed columns (arithmetic, `COALESCE`, `CASE WHEN`)

### Key things to remember about this database

- Film titles are **not unique** — always join/group by `film_id`, never
  by title text.
- `people` includes companies (`kind = 'person'` vs `'company'`) —
  filter when you only want humans.
- `title_zh` is partial (~1,324 of 5,265 films) — display fallback is
  `COALESCE(title_zh, title)`.
- `death_year` NULL is ambiguous: could mean alive, or unknown at IMDb.
- Honorary/SciTech nominations often have no linked film — that's by
  design, not missing data.

Full schema: `schema.sql` · Design rationale: `schema.md` ·
Source details: `data/data_notes.md`

### Reference docs

- SQLite SQL dialect: https://www.sqlite.org/lang.html
- SQLite tutorial: https://www.sqlitetutorial.net/
- pandas: https://pandas.pydata.org/docs/user_guide/index.html

## Part 0 — Setup

Connect to `oscars.db` and define one helper, `q()`, used for every query
in this notebook. `q()` runs SQL and returns a pandas DataFrame — the
result table you see under each code cell.

One cosmetic detail baked in from the start: pandas normally numbers
result rows starting at 0. `q()` shifts that to start at 1, purely for
readability (a "top 10" list reading 1–10 instead of 0–9). This never
touches the underlying data, only how row numbers are displayed.


In [1]:
import sqlite3
import pandas as pd

DB_PATH = "../oscars.db"  # relative to notebooks/

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")  # required every connection (SQLite quirk, see CLAUDE.md)

def q(sql, params=()):
    """Run a query and return a DataFrame, with row numbers starting at 1."""
    df = pd.read_sql_query(sql, conn, params=params)
    df.index = df.index + 1     # row numbers start at 1, not 0
    return df

# quick connection smoke test
q("SELECT COUNT(*) AS n FROM nominations;")

,n
1,12137


__Notes__

- **`sqlite3`** — Python's built-in SQLite driver; `sqlite3.connect(...)` opens
  (or creates) a database file and returns a connection object.
- **`PRAGMA foreign_keys = ON;`** — SQLite disables foreign-key enforcement
  by default, and it must be turned on for every new connection. Not a
  standard SQL statement — a SQLite-specific configuration command.
- **`pd.read_sql_query(sql, conn, params=params)`** — runs `sql` over
  connection `conn` and loads the result straight into a DataFrame, which
  is what renders as the nice HTML tables below each cell.
- **`COUNT(*)`** — an *aggregate function*: collapses all matching rows into
  a single number (a row count). Come back to aggregates in Part 4.
- **`AS n`** — an alias: renames the output column. Purely cosmetic —
  `COUNT(*)` alone would come back labeled `COUNT(*)`, awkward to reference;
  `AS n` calls it `n` instead.

Docs: https://www.sqlite.org/pragma.html#pragma_foreign_keys (PRAGMA foreign_keys) · https://www.sqlite.org/lang_aggfunc.html (aggregate functions)


## Part 1 — Reading from one table

**What we're learning:** the absolute basics — `SELECT`, `FROM`, `ORDER BY`,
`LIMIT`, and `DISTINCT`. Every query in this notebook builds on this
vocabulary, so it's worth being thorough here even though the queries
themselves are simple.


### 1.1 Selecting and ordering

__Q: What are all 98 Oscar ceremonies, oldest first?__

**Key syntax:** `SELECT`, `FROM`, `ORDER BY`

**Docs:** [SELECT](https://www.sqlite.org/lang_select.html) · [ORDER BY](https://www.sqlite.org/lang_select.html#the_order_by_clause)


In [2]:
q("""
SELECT ceremony, year_label      -- which columns to retrieve
FROM ceremonies                  -- which table to pull them from
ORDER BY ceremony;                -- sort ascending by ceremony number
""")

,ceremony,year_label
1,1,1927/28
2,2,1928/29
3,3,1929/30
4,4,1930/31
5,5,1931/32
...,...,...
94,94,2021
95,95,2022
96,96,2023
97,97,2024


__Notes__

- **`SELECT`** — lists which columns to retrieve, comma-separated.
  `SELECT *` would mean "all columns"; naming them explicitly (as here) is
  usually better practice, since it doesn't silently change if the table
  gains new columns later.
- **`FROM`** — names the table those columns come from. Every basic query
  needs a `SELECT` and a `FROM`; nothing runs without both.
- **`ORDER BY`** — sorts the result rows by a given column. Ascending by
  default (`1 → 98`); add `DESC` after the column name for descending.
  Without `ORDER BY`, SQL makes **no guarantee** about row order at all —
  not "unsorted," but *undefined* — so if order matters, always say so
  explicitly.
- **`;`** — marks the end of a SQL statement. Optional here since we're
  sending one query at a time, but required to separate multiple
  statements in a single string — worth keeping as a habit.


### 1.2 Distinct values

__Q: What award groups do we categorize nominations under?__

**Key syntax:** `DISTINCT`

**Docs:** [SELECT — DISTINCT](https://www.sqlite.org/lang_select.html#distinct)


In [3]:
q("SELECT DISTINCT award_group FROM categories ORDER BY award_group;")

,award_group
1,Actor -- Leading Role
2,Actor -- Supporting Role
3,Actress -- Leading Role
4,Actress -- Supporting Role
5,Animated Feature Film
6,Animated Short Film
7,Assistant Director
8,Best Picture
9,Casting
10,Cinematography


__Notes__

- **`DISTINCT`** — collapses duplicate values in the result down to one
  row each. Here it's almost redundant, since `categories.award_group` only
  has 37 rows total to begin with — but the pattern matters more once
  you're pulling distinct values out of a much bigger table (e.g. distinct
  `raw_category` values out of 12,137 `nominations` rows, later in Part 4).
- Without `DISTINCT`, `SELECT award_group FROM categories` would still only
  return 66 rows (one per category), since `categories` is already one row
  per `source_name` — `DISTINCT` only starts doing real work when the
  source table repeats values many times per group.


## Part 2 — Filtering rows

**What we're learning:** `WHERE` for filtering, `LIKE` for fuzzy text
matching, and how SQL represents "missing" — `NULL` — and the special
syntax needed to test for it.


### 2.1 Filtering with WHERE and pattern matching

__Q: The database has two different films called "Titanic" (1953 and
1997) — how do we find both?__

Real-world searches rarely match exactly what you type, so exact equality
(`title = 'Titanic'` would actually work here, but only by luck) is the
wrong tool to reach for — we want anything *containing* "Titanic."

**Key syntax:** `WHERE`, `LIKE`, `%` wildcard

**Docs:** [WHERE clause](https://www.sqlite.org/lang_select.html#the_where_clause) · [LIKE operator](https://www.sqlite.org/lang_expr.html#the_like_glob_regexp_and_match_operators)

In [4]:
q("""
SELECT film_id, title, release_year, imdb_id
FROM films
WHERE title LIKE '%Titanic%'
ORDER BY release_year;
""")

,film_id,title,release_year,imdb_id
1,1412,Titanic,1953,tt0046435
2,3685,Titanic,1997,tt0120338


__Notes__

- **`WHERE`** — filters *rows*: only rows where the condition is true make
  it into the result. Goes right after `FROM`.
- **`LIKE`** — a pattern-matching operator for text, used instead of `=`
  when you want partial or flexible matches rather than an exact one.
- **`%` wildcard** — inside a `LIKE` pattern, `%` matches any sequence of
  characters (including none). `'%Titanic%'` matches any title containing
  that substring anywhere in the string.
- **Case sensitivity** — SQLite's `LIKE` is case-insensitive for standard
  ASCII letters by default, so `'%titanic%'` would match the same rows.
- **Why this returns two rows, on purpose** — this is a live example of
  the "film titles aren't unique" rule (see the reminders at the top of
  this notebook): two different films, 1953 and 1997, share the exact
  same title. `film_id` and `imdb_id` are what tell them apart, not the
  title text — which is why every join or `GROUP BY` involving films
  elsewhere in this notebook groups by `film_id`, never by title.

### 2.2 Making a query reusable with parameters

__Q: Which people in the database have "Stewart" somewhere in their name?__

Same `LIKE` + `%` mechanism as 2.1, just applied to `people.name` instead
of `films.title` — but this time, instead of hardcoding the search text
into the SQL string, it's passed in as a variable. That's a small change
here, but it's the difference between a query you have to edit by hand
each time and one you can call repeatedly with a new search term.

**Key syntax:** `?` placeholder, `params=`

**Docs:** [Bind parameters](https://www.sqlite.org/lang_expr.html#varparam)

In [5]:
name_search = "%Stewart%"

result = q("""
SELECT person_id, name, birth_year, death_year, kind
FROM people
WHERE name LIKE ?          -- ? is a placeholder, filled in by params below
ORDER BY birth_year;
""", params=(name_search,))

# birth_year/death_year come back as float64 (NaN for missing) because the
# column mixes real years and NULLs -- Int64 keeps them as clean integers
# with <NA> instead of NaN. Purely a display fix, doesn't touch the data.
result["birth_year"] = result["birth_year"].astype("Int64")
result["death_year"] = result["death_year"].astype("Int64")
result

,person_id,name,birth_year,death_year,kind
1,1925,ROY C. STEWART AND SONS,<NA>,<NA>,person
2,2487,Stewart Filmscreen Corporation,<NA>,<NA>,company
3,3604,Edward Stewart,<NA>,<NA>,person
4,4056,Douglas Day Stewart,<NA>,<NA>,person
5,8134,Stewart Birnam,<NA>,<NA>,person
6,8753,Ross Stewart,<NA>,<NA>,person
7,10427,Leola Calzolai-Stewart,<NA>,<NA>,person
8,130,Donald Ogden Stewart,1894,1980,person
9,885,James G. Stewart,1907,1997,person
10,547,James Stewart,1908,1997,person


__Notes__

- **`?` and `params=`** — the query has a literal `?` placeholder instead
  of the value pasted directly into the SQL text; the actual value travels
  separately via `params=(name_search,)`. Two reasons this matters:
  - **Correctness** — a name with an apostrophe (`O'Brien`) would break a
    query built by pasting the raw string into the SQL text.
  - **Safety** — pasting untrusted text directly into SQL is how *SQL
    injection* happens; `?` + `params` is the standard defense, and the
    right habit to build before working with arbitrary user input.
  - `params` takes a **tuple** — the trailing comma in `(name_search,)` is
    required in Python to make it a one-element tuple; without it,
    `(name_search)` is just a parenthesized string, not a tuple.
- Same `LIKE` + `%` mechanism as 2.1, just on `people` instead of `films`,
  now with the search term as a parameter instead of hardcoded directly
  into the SQL.
- **Why this returns many rows, on purpose** — the point of this query is
  to surface *all* matches with enough distinguishing detail (`person_id`,
  `birth_year`, `kind`) to pick the right one before using it elsewhere
  (e.g. filtering by `person_id` instead of `name` in a later query).
- **`.astype("Int64")`** — see the code comment above; `Int64` (capital I,
  pandas' nullable integer type, distinct from NumPy's lowercase `int64`)
  displays whole years as whole numbers and missing values as `<NA>`
  instead of `NaN`.

### 2.3 Handling missing values

__Q: Which people in the database are missing a birth year?__

SQL's representation of "unknown" or "absent" is `NULL` — and it behaves
differently from an ordinary value, which trips people up constantly.

**Key syntax:** `IS NULL`, `IS NOT NULL`

**Docs:** [NULL handling](https://www.sqlite.org/lang_expr.html#isisnot)


In [6]:
q("""
SELECT name, kind
FROM people
WHERE birth_year IS NULL
  AND kind = 'person';
--  LIMIT 10;
""")

,name,kind
1,SIDNEY SANDERS,person
2,E. C. Wente,person
3,A.E. Kaye,person
4,JOSEPH E. ROBBINS,person
5,Marcella Burke,person
...,...,...
4047,Michael Simkin,person
4048,Neil Goodwin,person
4049,Phil Morrison,person
4050,Helen Ryan Dobrowski,person


__Notes__

- **`NULL`** means "unknown" or "absent" — never zero, never an empty
  string. In this database it shows up whenever a value genuinely isn't
  known (e.g. `birth_year` for someone with no IMDb record).
- **`IS NULL` / `IS NOT NULL`**, not `= NULL` / `!= NULL` — this is the
  detail that catches almost everyone once. `NULL` represents "unknown," so
  asking `birth_year = NULL` is really asking "is this unknown value equal
  to unknown?" — which SQL correctly refuses to answer `TRUE` or `FALSE` to;
  it returns `NULL` itself, and a `WHERE` clause treats a `NULL` result as
  "not a match." The special `IS` / `IS NOT` forms exist specifically to
  test for NULL-ness without falling into that trap.
- We'll use `IS NOT NULL` again in Part 8, to exclude people with no
  `birth_year` before doing arithmetic on that column — arithmetic against
  `NULL` just produces another `NULL`, silently, rather than an error.
- **`kind = 'person'`** — companies structurally never have a `birth_year`
  (they're not people), so without this filter the query would mostly
  rediscover "companies exist" rather than surface people whose birth year
  is genuinely unknown.


## Part 3 — Combining tables

**What we're learning:** `JOIN`, the single most important piece of SQL
for a relational database like this one — almost every interesting
question needs data from more than one table.


### 3.1 Combining two tables

__Q: What film won Best Picture in 2019?__

`nominations` alone doesn't have a year label or an award-group name or a
film title — those live in `ceremonies`, `categories`, and `films`
respectively. `JOIN` is how you pull columns from other tables into one
result.

**Key syntax:** `JOIN ... ON ...`, table aliases, chaining multiple joins

**Docs:** [JOIN clause](https://www.sqlite.org/syntax/join-clause.html)


In [7]:
q("""
SELECT c.year_label, cat.award_group, f.title
FROM nominations n
JOIN ceremonies c ON n.ceremony = c.ceremony              -- brings in year_label
JOIN categories cat ON n.category_id = cat.category_id    -- brings in award_group
JOIN nomination_films nf ON n.nomination_id = nf.nomination_id  -- junction table
JOIN films f ON nf.film_id = f.film_id                    -- brings in title
WHERE n.is_winner = 1
  AND cat.award_group = 'Best Picture'
  AND c.year_label = '2019';
""")

,year_label,award_group,title
1,2019,Best Picture,Parasite


__Notes__

- **`JOIN ... ON ...`** — combines two tables by matching rows where a
  column in one equals a column in the other.
  `JOIN ceremonies c ON n.ceremony = c.ceremony` reads as: "for each
  nomination row, find the ceremony row whose `ceremony` value matches, and
  stick their columns together." Without this, `nominations` alone has no
  `year_label` or `award_group` at all.
- **Chained joins** — you can `JOIN` more than once in one query, each
  bringing in another table. Here: `ceremonies` (year), `categories`
  (award-group name), `nomination_films` (the junction table linking a
  nomination to its film(s) — see below), then `films` (title). The order
  of `JOIN` clauses doesn't affect the result — SQLite figures out an
  efficient plan regardless of the order you write them in.
- **Junction tables** — a nomination can have multiple films (or none),
  and a film can appear in multiple nominations, so the link between them
  isn't a plain foreign key — it's a separate table, `nomination_films`,
  with one row per (nomination, film) pair. Joining through it is a
  two-step hop: `nominations → nomination_films → films`.
- **Table aliases** (`n`, `c`, `cat`, `nf`, `f`) — short names assigned
  right after each table name (`nominations n` = "call this table `n` from
  now on"). Necessary once multiple tables are involved: both
  `nominations` and `categories` have a `category_id`-ish column, so
  `n.category_id` vs `cat.category_id` needs the alias to stay
  unambiguous.
- **`WHERE` after the joins** — filtering happens on the *combined* table,
  after all the joins have run; that's why `WHERE` here can reference
  columns from `ceremonies` and `categories`, not just `nominations`.


### 3.2 Chained joins with a parameter

__Q: What is Daniel Day-Lewis's complete Oscar nomination history?__

Same join shape as 3.1, extended one hop further (through
`nomination_people`, the person-side junction table), and this time the
filter value is a variable rather than a literal — introducing
parameterized queries.

**Key syntax:** join chain through a junction table, `?` placeholder + `params`

**Docs:** [Bind parameters](https://www.sqlite.org/lang_expr.html#varparam)


In [8]:
person_name = "Daniel Day-Lewis"

q("""
SELECT c.year_label, cat.award_group, n.official_name, n.is_winner
FROM nominations n
JOIN nomination_people np ON n.nomination_id = np.nomination_id  -- junction table
JOIN people p ON np.person_id = p.person_id                     -- brings in the person
JOIN ceremonies c ON n.ceremony = c.ceremony
JOIN categories cat ON n.category_id = cat.category_id
WHERE p.name = ?
ORDER BY c.ceremony;
""", params=(person_name,))

,year_label,award_group,official_name,is_winner
1,1989,Actor -- Leading Role,Daniel Day Lewis,1
2,1993,Actor -- Leading Role,Daniel Day-Lewis,0
3,2002,Actor -- Leading Role,Daniel Day-Lewis,0
4,2007,Actor -- Leading Role,Daniel Day-Lewis,1
5,2012,Actor -- Leading Role,Daniel Day-Lewis,1
6,2017,Actor -- Leading Role,Daniel Day-Lewis,0


__Notes__

- **Same join pattern as 3.1**, applied to people instead of films:
  `nominations` → junction table (`nomination_people`) → entity table
  (`people`), plus `ceremonies`/`categories` joined in for display columns.
- **`?` and `params=`** — see 2.1's notes for the full explanation
  (correctness with special characters, safety against SQL injection).
  Here the placeholder stands in for a person's name instead of a search
  pattern.
- **Filtering on one dimension, returning many rows** — `WHERE p.name = ?`
  pins down only the person, leaving category and year unconstrained, so
  the result is a full career history rather than a single row — which is
  why `ORDER BY c.ceremony` matters here.
- **Known limitation** — `p.name = ?` assumes exactly one `people` row
  matches this name. Names aren't guaranteed unique in this dataset;
  disambiguating properly is what the 2.1 pattern (search first, then
  filter by `person_id`) is for.


### 3.3 Keeping rows with no match

__Q: Which directors have never received an Oscar nomination themselves?__

Every `JOIN` used so far (3.1, 3.2) is an **inner join**: if a row on one
side has no matching row on the other, it silently disappears from the
result. That's fine when you want only matched pairs — but here we want
the opposite: directors from `film_directors` who *don't* show up
anywhere in `nomination_people`. An inner join can't produce that; a
`LEFT JOIN` can.

**Key syntax:** `LEFT JOIN`, combined with `IS NULL` (from 2.3) to find
the non-matches

**Docs:** [LEFT JOIN](https://www.sqlite.org/syntax/join-clause.html)

In [21]:
result = q("""
SELECT DISTINCT p.name
FROM film_directors fd
JOIN people p ON fd.person_id = p.person_id
LEFT JOIN nomination_people np ON p.person_id = np.person_id   -- keep every director, matched or not
WHERE np.person_id IS NULL                                     -- keep only the ones with no match
ORDER BY p.name
LIMIT 10;
""")

result

,name
1,A. Edward Sutherland
2,Aaron Schimberg
3,Abderrahmane Sissako
4,Abe Levitow
5,Adolfo Aristarain
6,Adriana Bosch
7,Agnès Jaoui
8,Aida Zyablikova
9,Aki Kaurismäki
10,Al Teeter


__Notes__

- **`LEFT JOIN`** — like `JOIN` (3.1), it matches rows between two tables
  on a condition — but where a plain `JOIN` *drops* a left-side row that
  has no match on the right, `LEFT JOIN` *keeps* it anyway, filling in
  `NULL` for every column that would have come from the right-hand table.
  "Left" refers to which table is written first (`film_directors`, via
  `people`) — that side's rows are always preserved.
- **`WHERE np.person_id IS NULL`** — this is the trick that makes
  `LEFT JOIN` useful for "find the non-matches" questions. Only rows where
  `nomination_people` had *nothing* to match end up with `np.person_id`
  actually `NULL` (from the previous bullet); filtering on that
  `IS NULL`, from 2.3, keeps only directors whose row never found a
  partner — i.e. never appear in `nomination_people` at all. This
  combination (`LEFT JOIN` + `IS NULL` on the joined-in column) is a common
  enough pattern that it has a name: an **anti-join**.
- **Why `DISTINCT` is needed here** — `film_directors` has one row per
  (film, director) pair, so a prolific director shows up multiple times
  (once per film they directed). Without `DISTINCT`, the same never-
  nominated director's name could print several times over.
- **A large result, on purpose** — per `data_notes.md`, 1,508 people exist
  in `people` *only* because they directed a film and were never
  personally nominated for anything; this query surfaces exactly that
  population (`LIMIT 10` above just keeps the display short — drop it to
  see the full list).
- **Contrast with 3.1/3.2** — every earlier join in this notebook only
  ever showed matched pairs. This is the first query where the *absence*
  of a match is the actual point of the question.

## Part 4 — Aggregating

**What we're learning:** turning many rows into summary numbers —
`GROUP BY`, `COUNT`, `SUM`, `HAVING`, `MIN`/`MAX` — and the crucial
distinction between filtering individual rows (`WHERE`) and filtering
already-grouped results (`HAVING`).


### 4.1 Counting within groups

__Q: What are the 10 most-awarded films of all time?__

**Key syntax:** `GROUP BY`, `COUNT(*)`, `LIMIT` for a "top N" pattern

**Docs:** [GROUP BY](https://www.sqlite.org/lang_select.html#the_group_by_clause) · [LIMIT](https://www.sqlite.org/lang_select.html#the_limit_clause)


In [9]:
q("""
SELECT f.title, f.release_year, COUNT(*) AS wins
FROM films f
JOIN nomination_films nf ON f.film_id = nf.film_id
JOIN nominations n ON nf.nomination_id = n.nomination_id
WHERE n.is_winner = 1
GROUP BY f.film_id       -- one row per film, not per title (see notes)
ORDER BY wins DESC
LIMIT 10;
""")

,title,release_year,wins
1,The Lord of the Rings: The Return of the King,2003,11
2,Titanic,1997,11
3,Ben-Hur,1959,11
4,West Side Story,1961,10
5,Gone with the Wind,1939,10
6,The English Patient,1996,9
7,The Last Emperor,1987,9
8,Gigi,1958,9
9,Slumdog Millionaire,2008,8
10,Amadeus,1984,8


__Notes__

- **`GROUP BY`** — collapses multiple rows into one row *per distinct
  value* of the grouped column. `GROUP BY f.film_id` means: instead of one
  row per winning nomination, give one row per film, with everything else
  (like `COUNT(*)`) computed *within* that film's group of rows.
- **`GROUP BY f.film_id`, not `f.title`** — the film-titles-aren't-unique
  rule from Part 2, in action. Two different films are both named
  "Titanic"; grouping by title would silently merge their win counts into
  one row. Grouping by the surrogate `film_id` keeps them separate, since
  IDs are guaranteed unique.
- **`COUNT(*)` means something different here than in Part 0** — in Part
  0's version there was no `GROUP BY`, so `COUNT(*)` collapsed the *entire
  table* into one number. With `GROUP BY`, `COUNT(*)` instead counts rows
  *within each group* — here, how many winning-nomination rows exist per
  film. Same function, different scope, depending on whether `GROUP BY` is
  present.
- **`LIMIT 10`** — caps the result to the first 10 rows, applied after
  `ORDER BY` sorts them. Without it, every film with at least one win would
  come back — hundreds of rows.
- **A subtlety worth sitting with:** a column in `SELECT` (like `f.title`,
  `f.release_year`) that isn't in `GROUP BY` and isn't wrapped in an
  aggregate function is only safe to select if it's *functionally
  dependent* on the grouped column — every row within a `film_id` group
  necessarily has the same title and release year, so there's no ambiguity
  about which value to show. SQLite allows this; some stricter databases
  (e.g. PostgreSQL) would reject it unless you add those columns to
  `GROUP BY` too, or wrap them in an aggregate like `MIN()`.


### 4.2 Filtering groups, not rows

__Q: Who has the most Oscar nominations without ever winning?__

**Key syntax:** `SUM` on a 0/1 flag column, `HAVING` vs `WHERE`

**Docs:** [HAVING clause](https://www.sqlite.org/lang_select.html#the_having_clause)


In [10]:
q("""
SELECT p.name, COUNT(*) AS nominations, SUM(n.is_winner) AS wins
FROM people p
JOIN nomination_people np ON p.person_id = np.person_id
JOIN nominations n ON np.nomination_id = n.nomination_id
WHERE p.kind = 'person'          -- a per-row fact, filters before grouping
GROUP BY p.person_id
HAVING SUM(n.is_winner) = 0      -- a per-group fact, filters after grouping
ORDER BY nominations DESC
LIMIT 10;
""")

,name,nominations,wins
1,Greg P. Russell,16,0
2,Thomas Newman,15,0
3,Roland Anderson,15,0
4,George J. Folsey,14,0
5,Daniel Sudick,13,0
6,Bradley Cooper,12,0
7,Rick Kline,11,0
8,Anna Behlmer,10,0
9,Walter Scharf,10,0
10,Ren Klyce,9,0


__Notes__

- **`SUM(n.is_winner)`** — `is_winner` is stored as 0/1, so summing it
  across a group counts how many of a person's nominations were wins. A new
  use of an aggregate: 4.1 only used `COUNT(*)` to count rows; this sums a
  flag column instead.
- **`HAVING` vs `WHERE`** — the key distinction in this query. `WHERE`
  filters individual rows *before* any grouping happens; `HAVING` filters
  *groups*, after `GROUP BY` has already collapsed rows. That's why
  `WHERE p.kind = 'person'` (a per-row fact) works fine as `WHERE`, but
  `HAVING SUM(n.is_winner) = 0` (a per-group fact) has to be `HAVING` —
  writing `WHERE SUM(n.is_winner) = 0` would be a SQL error, since
  aggregates don't exist yet at the point `WHERE` is evaluated.
- **Execution order, not written order** — SQL is *written*
  `SELECT ... FROM ... WHERE ... GROUP BY ... HAVING ... ORDER BY`, but
  *executes* `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY`.
  That's the whole reason `HAVING` can reference `SUM(...)` and `WHERE`
  can't — by the time `HAVING` runs, the sums already exist.
- **`p.kind = 'person'`** — excludes company/organization rows, since
  "nominated a lot but never won" is only a meaningful question for
  individuals.


### 4.3 Counting distinct values within a group

__Q: Who has won across the most different Oscar categories, and which ones?__

**Key syntax:** `COUNT(DISTINCT ...)`, `GROUP_CONCAT(DISTINCT ...)`

**Docs:** [COUNT with DISTINCT](https://www.sqlite.org/lang_aggfunc.html#count) · [GROUP_CONCAT](https://www.sqlite.org/lang_aggfunc.html#group_concat)

In [11]:
result = q("""
SELECT p.name,
       COUNT(*) AS wins,
       COUNT(DISTINCT n.category_id) AS distinct_categories,
       GROUP_CONCAT(DISTINCT cat.award_group) AS categories
FROM people p
JOIN nomination_people np ON p.person_id = np.person_id
JOIN nominations n ON np.nomination_id = n.nomination_id
JOIN categories cat ON n.category_id = cat.category_id
WHERE p.kind = 'person'
  AND n.is_winner = 1
GROUP BY p.person_id
ORDER BY distinct_categories DESC, wins DESC
LIMIT 10;
""")

# GROUP_CONCAT can return long strings that pandas truncates by default;
# this shows the full text for just this result, without changing display
# behavior anywhere else in the notebook.
with pd.option_context('display.max_colwidth', None):
    display(result)

,name,wins,distinct_categories,categories
1,Walt Disney,26,7,"Animated Short Film,Honorary Award,Irving G. Thalberg Memorial Award,Live Action Short Film,Documentary Feature Film,Documentary Short Film"
2,Douglas Shearer,14,5,"Sound Mixing,Scientific and Technical (Technical Achievement Award),Scientific and Technical (Academy Award of Merit),Scientific and Technical (Scientific and Engineering Award),Visual Effects"
3,Loren L. Ryder,8,5,"Honorary Award,Scientific and Technical (Technical Achievement Award),Scientific and Technical (Scientific and Engineering Award),Scientific and Technical (Academy Award of Merit),Scientific and Technical (Bonner Medal)"
4,Billy Wilder,7,5,"Directing,Writing,Best Picture,Irving G. Thalberg Memorial Award"
5,Jonathan Erland,7,5,"Scientific and Technical (Scientific and Engineering Award),Scientific and Technical (Technical Achievement Award),Scientific and Technical (Bonner Medal),Scientific and Technical (Gordon E. Sawyer Award)"
6,Francis Ford Coppola,6,5,"Writing,Directing,Best Picture,Irving G. Thalberg Memorial Award"
7,Petro Vlahos,5,5,"Scientific and Technical (Technical Achievement Award),Scientific and Technical (Academy Award of Merit),Scientific and Technical (Bonner Medal),Scientific and Technical (Gordon E. Sawyer Award)"
8,Farciot Edouart,10,4,"Scientific and Technical (Scientific and Engineering Award),Honorary Award,Scientific and Technical (Technical Achievement Award),Visual Effects"
9,Richard Edlund,8,4,"Visual Effects,Special Achievement Award,Scientific and Technical (Scientific and Engineering Award),Scientific and Technical (Bonner Medal)"
10,Gordon Hollingshead,7,4,"Assistant Director,Live Action Short Film,Documentary Short Film"


__Notes__

- **`COUNT(DISTINCT n.category_id)`** — a new use of `COUNT`: `DISTINCT`
  first collapses duplicate values before counting, so this counts how
  many *different* categories a person has won in, not how many wins
  total. Two people could tie on win count but differ hugely in
  "breadth" — one winning 10 times in one category, another winning across
  10 different ones — and this column is what tells them apart.
- **`GROUP_CONCAT(DISTINCT cat.award_group)`** — an aggregate that joins
  every value in a group into one comma-separated string, instead of
  collapsing them to a count. `DISTINCT` here avoids listing the same
  category twice if someone won it more than once. Default separator is a
  comma; `GROUP_CONCAT(expr, ' | ')` would use a custom one instead.
- **`ORDER BY distinct_categories DESC, wins DESC`** — sorts primarily by
  category breadth (what the question actually asks), using win count only
  to break ties between people with the same number of distinct
  categories.
- **`WHERE n.is_winner = 1`, not `HAVING`** — looks similar to 4.2 but
  works differently, worth being precise about: here `WHERE` excludes
  losing-nomination rows *before* grouping even happens, so only winning
  rows ever enter a person's group, meaning `COUNT(*)` directly equals win
  count. 4.2 needed the opposite — *all* nomination rows (wins and losses)
  had to enter each group first, so `COUNT(*)` could total nominations and
  a separate `SUM(n.is_winner)` could total wins, with `HAVING` checking
  the result afterward. Same clauses, opposite reason for using them:
  `WHERE` here narrows the input; `HAVING` in 4.2 checked an output.
- **`p.kind = 'person'`** — same reasoning as 4.2: excludes
  company/organization rows, since "who's won across the most categories"
  is a question about individuals.
- **Long strings get truncated in the display** — pandas clips wide cell
  values by default (`display.max_colwidth`). `pd.option_context(...)`
  temporarily overrides that setting only for the code inside the `with`
  block, then reverts automatically — cleaner than `pd.set_option(...)`,
  which would change the setting for every cell afterward in the
  notebook. `display(result)` is needed instead of a bare `result` on the
  last line, since the `with` block changes how the notebook decides
  *when* to render, not Jupyter's normal "show the last expression"
  behavior.

### 4.4 Earliest and latest within a group

__Q: How did the category name for Best Actor change over the decades?__

**Key syntax:** `MIN()` / `MAX()` as aggregates, grouping on free text

**Docs:** [MIN/MAX](https://www.sqlite.org/lang_aggfunc.html#minoraggmax)


In [12]:
category_of_interest = "ACTOR IN A LEADING ROLE"

q("""
SELECT n.raw_category, MIN(n.ceremony) AS first_ceremony, MAX(n.ceremony) AS last_ceremony, COUNT(*) AS times_used
FROM nominations n
JOIN categories cat ON n.category_id = cat.category_id
WHERE cat.source_name = ?    -- filter on the stable join key
GROUP BY n.raw_category      -- group on the free-text label as written that year
ORDER BY first_ceremony;
""", params=(category_of_interest,))

,raw_category,first_ceremony,last_ceremony,times_used
1,ACTOR,1,48,232
2,ACTOR IN A LEADING ROLE,49,98,250


__Notes__

- **`MIN(...)` / `MAX(...)`** — same aggregate-function family as `COUNT`
  and `SUM` (4.1, 4.2): instead of counting or totaling, they return the
  smallest/largest value seen within a group. Here, the earliest and
  latest ceremony a given raw-text category name was used.
- **Filtering on `source_name`, grouping on `raw_category`** —
  `source_name` is the stable join key (66 fixed values, per the category
  hierarchy in `schema.md`); `raw_category` is the free-text label as
  written that specific year, which can vary even while `source_name`
  stays fixed. Filtering on the stable column and grouping on the variable
  one is what surfaces historical wording changes — here, the shift from
  the early plain `"ACTOR"` to the modern `"ACTOR IN A LEADING ROLE"`.
- **`GROUP BY n.raw_category` collapsing repeated years** — every
  ceremony that used a given wording collapses into one row for that
  wording, with `MIN`/`MAX`/`COUNT` summarizing across however many
  ceremonies used it.


## Part 5 — Set membership

**What we're learning:** `IN`, for checking membership against a list of
values, and how to build that list dynamically from Python rather than
hardcoding it.


### 5.1 Matching against a list of values

__Q: Who has the most wins combined across Cinematography and Film
Editing?__

**Key syntax:** `IN (...)`, dynamically generating one `?` per list item

**Docs:** [IN operator](https://www.sqlite.org/lang_expr.html#in_op)


In [13]:
categories_of_interest = ["Cinematography", "Film Editing"]

# build one '?' placeholder per category, since the list length can vary
placeholders = ", ".join("?" for _ in categories_of_interest)

result = q(f"""
SELECT p.name, COUNT(*) AS wins
FROM people p
JOIN nomination_people np ON p.person_id = np.person_id
JOIN nominations n ON np.nomination_id = n.nomination_id
JOIN categories cat ON n.category_id = cat.category_id
WHERE p.kind = 'person'
  AND n.is_winner = 1
  AND cat.award_group IN ({placeholders})
GROUP BY p.person_id
ORDER BY wins DESC
LIMIT 10;
""", params=tuple(categories_of_interest))

result

,name,wins
1,Leon Shamroy,4
2,Joseph Ruttenberg,4
3,Emmanuel Lubezki,3
4,Robert Richardson,3
5,Vittorio Storaro,3
6,Michael Kahn,3
7,Thelma Schoonmaker,3
8,Conrad L. Hall,3
9,Freddie Young,3
10,Robert Surtees,3


__Notes__

- **`IN (...)`** — matches if a value equals any item in a list; shorthand
  for chaining several `OR cat.award_group = ...` conditions.
- **`", ".join("?" for _ in categories_of_interest)`** — builds a `?, ?, ...`
  placeholder string with one `?` per category, since the count varies with
  the list length. Still fully parameterized: only the *placeholder count*
  is built in Python; the actual category names travel through `params`,
  never pasted directly into the SQL text.
- **f-string vs `params` — two different jobs** — the f-string fills in
  query *structure* (how many placeholders) before SQLite ever sees the
  string; `params` fills in query *data* (the actual values) safely, after
  the fact. Never use an f-string to insert a *value* directly into SQL —
  that reopens the SQL-injection risk `?` + `params` exists to close.
- **Exact string matching** — `award_group` values must match precisely,
  including case and wording (`'Cinematography'`, not `'CINEMATOGRAPHY'` or
  `'Best Cinematography'`). Run `SELECT DISTINCT award_group FROM
  categories;` (section 1.2) to get exact values rather than guessing.


## Part 6 — Subqueries

**What we're learning:** a query nested inside another query's `WHERE`
clause, used to answer "does this row appear in some other set?" This part
builds up in layers — each question adds one more piece of subquery logic
on top of the last.


### 6.1 Excluding rows found by another query

__Q: Which Best Picture winners were directed by someone never nominated
for Directing themselves?__

**Key syntax:** `NOT IN` with a subquery

**Docs:** [Subqueries with IN/NOT IN](https://www.sqlite.org/lang_expr.html#in_op)

In [14]:
result = q("""
SELECT p.name, f.title, c.year_label
FROM people p
JOIN film_directors fd ON p.person_id = fd.person_id
JOIN films f ON fd.film_id = f.film_id
JOIN nomination_films nf ON f.film_id = nf.film_id
JOIN nominations n ON nf.nomination_id = n.nomination_id
JOIN categories cat ON n.category_id = cat.category_id
JOIN ceremonies c ON n.ceremony = c.ceremony
WHERE n.is_winner = 1
  AND cat.award_group = 'Best Picture'
  AND p.person_id NOT IN (
      -- subquery: everyone ever nominated in the Directing group
      SELECT np.person_id
      FROM nomination_people np
      JOIN nominations n2 ON np.nomination_id = n2.nomination_id
      JOIN categories cat2 ON n2.category_id = cat2.category_id
      WHERE cat2.award_group = 'Directing'
  )
ORDER BY c.ceremony;
""")

result

,name,title,year_label
1,Edmund Goulding,Grand Hotel,1931/32
2,Loveleen Tandan,Slumdog Millionaire,2008
3,Ben Affleck,Argo,2012
4,Peter Farrelly,Green Book,2018
5,Sian Heder,CODA,2021


__Notes__

- **Subquery with `NOT IN`** — the inner query (in parentheses) runs
  first and produces a list of `person_id`s: everyone ever nominated in the
  Directing group. The outer query keeps only rows where the director's
  `person_id` isn't in that list.
- **Why a subquery, not a join-based exclusion** — "never nominated for
  Directing" is a fact about a person's *entire* nomination history, not
  something expressible within the outer query's own join chain (built
  around Best-Picture-winning films). The subquery answers a separate
  question — "who has ever appeared in this other set?" — and hands back a
  list to filter against.
- **Two independent alias sets (`n`/`cat`/`c` vs `n2`/`cat2`)** — the
  inner and outer queries each join `nominations`/`categories` for
  different purposes (outer: which film won Best Picture; inner: who's
  ever been nominated for Directing), so distinct aliases are needed to
  avoid colliding within one statement.
- **`film_directors`, independent of `nominations`** — director credit
  here comes from IMDb crew data, not from being nominated as "Director" in
  `nominations`. That's why a director can appear in this result at all:
  they directed the film (via `film_directors`), but their own name may
  never appear in `nomination_people` for the Directing group.
- **`award_group = 'Directing'`** — the hierarchy level that groups the
  real Directing category with its historical Comedy/Dramatic splits,
  while excluding Assistant Director (a defunct, different job). Choosing
  the right hierarchy level matters: `class = 'Directing'` would sweep
  Assistant Director in too.

### 6.2 A single computed value inside the query

__Q: Of the last 25 Best Picture winners, which had no Film Editing
nomination?__

**Key syntax:** scalar subquery — a subquery that returns exactly one
value, used directly in arithmetic

**Docs:** [Scalar subqueries](https://www.sqlite.org/lang_expr.html#scalar_subqueries)


In [15]:
recent_n = 25   # number of most recent ceremonies to consider

result = q("""
SELECT f.title, c.year_label
FROM films f
JOIN nomination_films nf ON f.film_id = nf.film_id
JOIN nominations n ON nf.nomination_id = n.nomination_id
JOIN categories cat ON n.category_id = cat.category_id
JOIN ceremonies c ON n.ceremony = c.ceremony
WHERE n.is_winner = 1
  AND cat.award_group = 'Best Picture'
  AND c.ceremony > (SELECT MAX(ceremony) FROM ceremonies) - ?   -- scalar subquery
  AND f.film_id NOT IN (
      SELECT nf2.film_id
      FROM nomination_films nf2
      JOIN nominations n2 ON nf2.nomination_id = n2.nomination_id
      JOIN categories cat2 ON n2.category_id = cat2.category_id
      WHERE cat2.award_group = 'Film Editing'
  )
ORDER BY c.ceremony;
""", params=(recent_n,))

result

,title,year_label
1,Birdman or (The Unexpected Virtue of Ignorance),2014
2,CODA,2021


__Notes__

- **Filtering the film, not the person** — 6.1 excluded rows where the
  *director* appeared in a certain nomination list; this excludes rows
  where the *film itself* appeared in one. Same `NOT IN` mechanism, applied
  to a different foreign key. (The Film Editing / Best Picture correlation
  is well-known Oscar trivia — BP winners almost always have an editing
  nomination — so this result should be short.)
- **Scalar subquery in `WHERE`** — `(SELECT MAX(ceremony) FROM ceremonies)`
  returns a single value (the highest ceremony number), then used directly
  in an arithmetic comparison. This makes the "past 25" cutoff
  future-proof: no hardcoded `>= 74`; when a 99th ceremony is added, the
  query still means "past 25."
- **A parameter inside arithmetic** — the `?` placeholder can appear
  inside an expression (`- ?`), not just as a whole standalone value. The
  parameter fills in exactly like a literal number would.


### 6.3 Excluding against two conditions at once

__Q: Which Best Picture winners had *neither* a Directing nor a Film
Editing nomination?__

**Key syntax:** `IN` used inside a `NOT IN` subquery, to widen what gets
excluded

**Docs:** [IN operator](https://www.sqlite.org/lang_expr.html#in_op)


In [16]:
result = q("""
SELECT f.title, c.year_label
FROM films f
JOIN nomination_films nf ON f.film_id = nf.film_id
JOIN nominations n ON nf.nomination_id = n.nomination_id
JOIN categories cat ON n.category_id = cat.category_id
JOIN ceremonies c ON n.ceremony = c.ceremony
WHERE n.is_winner = 1
  AND cat.award_group = 'Best Picture'
  AND f.film_id NOT IN (
      SELECT nf2.film_id
      FROM nomination_films nf2
      JOIN nominations n2 ON nf2.nomination_id = n2.nomination_id
      JOIN categories cat2 ON n2.category_id = cat2.category_id
      WHERE cat2.award_group IN ('Directing', 'Film Editing')   -- either group
  )
ORDER BY c.ceremony;
""")

result

,title,year_label
1,Wings,1927/28
2,Grand Hotel,1931/32
3,CODA,2021


__Notes__

- **`NOT IN` outside, `IN` inside** — the outer `NOT IN` excludes films
  that appear in the inner subquery's result; the inner
  `IN ('Directing', 'Film Editing')` widens the subquery to pull
  `film_id`s nominated in *either* group. Together: exclude films
  nominated for either Directing or Editing, meaning the survivors had
  *neither*.
- **Why one combined subquery, not two separate `NOT IN`s** — one
  subquery collecting film IDs from both groups is cleaner than chaining
  `AND f.film_id NOT IN (Directing subquery) AND f.film_id NOT IN (Editing
  subquery)`. Both give the same answer, but the single-subquery form only
  reads `nomination_films`/`nominations`/`categories` once inside the
  exclusion, and reads as one intent — "exclude films with either kind of
  nomination" — rather than two.
- **Why `IN`, not `OR`** — `IN ('Directing', 'Film Editing')` is the
  idiomatic way to say "equals any of these values"; the equivalent
  `award_group = 'Directing' OR award_group = 'Film Editing'` works too but
  scales worse and reads less clearly as the list grows.


### 6.4 Mixing OR/AND and different levels of the category hierarchy

__Q: Which Best Picture winners had no Directing nomination *and* no
Acting win?__

**Key syntax:** combining `OR` + `AND` with explicit parentheses; mixing
`award_group` and `class` in one subquery

**Docs:** [Operator precedence](https://www.sqlite.org/lang_expr.html#operators_and_parse_affecting_attributes_of_various_operators)


In [17]:
result = q("""
SELECT f.title, c.year_label
FROM films f
JOIN nomination_films nf ON f.film_id = nf.film_id
JOIN nominations n ON nf.nomination_id = n.nomination_id
JOIN categories cat ON n.category_id = cat.category_id
JOIN ceremonies c ON n.ceremony = c.ceremony
WHERE n.is_winner = 1
  AND cat.award_group = 'Best Picture'
  AND f.film_id NOT IN (
      SELECT nf2.film_id
      FROM nomination_films nf2
      JOIN nominations n2 ON nf2.nomination_id = n2.nomination_id
      JOIN categories cat2 ON n2.category_id = cat2.category_id
      WHERE cat2.award_group = 'Directing'
         OR (cat2.class = 'Acting' AND n2.is_winner = 1)
  )
ORDER BY c.ceremony;
""")

result

,title,year_label
1,Wings,1927/28
2,Grand Hotel,1931/32
3,Argo,2012


__Notes__

- **Mixing hierarchy levels within one subquery** —
  `cat2.award_group = 'Directing'` narrowly catches the Directing category
  and its historical Comedy/Dramatic splits (excluding Assistant Director);
  `cat2.class = 'Acting'` broadly catches all four acting categories at
  once. Choosing different hierarchy levels for different filter
  conditions is fine, and often clearer than expanding one of them into a
  long list at a finer level.
- **`OR` here, not `IN`** — the two conditions target *different columns*
  (`award_group` vs `class`). `IN` compares one column to a list of values;
  `OR` combines conditions across columns.
- **Asymmetric conditions: nomination vs. win** — the Directing side
  excludes films with *any* Directing nomination; the Acting side excludes
  only films that *won* an Acting award, letting acting-nominated-but-lost
  films through. Achieved by attaching `AND n2.is_winner = 1` to the Acting
  condition specifically.
- **Parenthesizing `(cat2.class = 'Acting' AND n2.is_winner = 1)`** — SQL
  evaluates `AND` before `OR` by default (like `*` before `+`), so these
  parentheses don't change the meaning here, but they make the intended
  grouping explicit. Once `OR` and `AND` mix in one condition, wrapping the
  intended grouping beats trusting operator precedence to be read
  correctly later.


## Part 7 — Alternative patterns for the same job

**What we're learning:** SQL often has more than one correct way to write
the same question. This part revisits queries from Part 6 using different
syntax — partly to show the alternatives, partly because each alternative
has a real advantage the `NOT IN` version doesn't.


### 7.1 Checking existence instead of membership

__Q: Same question as 6.1 — which Best Picture winners were directed by
someone never nominated for Directing? — solved a different way.__

`NOT IN` (6.1) builds a full list of every Directing-nominated
`person_id`, then checks whether the outer row's ID is missing from that
list. `NOT EXISTS` asks a narrower question directly: "does *any* row
exist matching this specific person?" — without ever materializing the
full list. Same answer, different mechanism, and one that avoids a real
`NOT IN` pitfall (see notes).

**Key syntax:** `EXISTS` / `NOT EXISTS`, correlated subqueries

**Docs:** [EXISTS](https://www.sqlite.org/lang_expr.html#the_exists_operator)

In [23]:
result = q("""
SELECT p.name, f.title, c.year_label
FROM people p
JOIN film_directors fd ON p.person_id = fd.person_id
JOIN films f ON fd.film_id = f.film_id
JOIN nomination_films nf ON f.film_id = nf.film_id
JOIN nominations n ON nf.nomination_id = n.nomination_id
JOIN categories cat ON n.category_id = cat.category_id
JOIN ceremonies c ON n.ceremony = c.ceremony
WHERE n.is_winner = 1
  AND cat.award_group = 'Best Picture'
  AND NOT EXISTS (
      SELECT 1                                   -- the actual column here is irrelevant
      FROM nomination_people np
      JOIN nominations n2 ON np.nomination_id = n2.nomination_id
      JOIN categories cat2 ON n2.category_id = cat2.category_id
      WHERE cat2.award_group = 'Directing'
        AND np.person_id = p.person_id           -- ties the subquery to the outer row
  )
ORDER BY c.ceremony;
""")

result

,name,title,year_label
1,Edmund Goulding,Grand Hotel,1931/32
2,Loveleen Tandan,Slumdog Millionaire,2008
3,Ben Affleck,Argo,2012
4,Peter Farrelly,Green Book,2018
5,Sian Heder,CODA,2021


__Notes__

- **`EXISTS` / `NOT EXISTS`** — returns true or false depending on whether
  the subquery produces *any* row at all; it never looks at what's in
  those rows, just whether they exist. `NOT EXISTS` is true when the
  subquery produces nothing.
- **`SELECT 1` inside the subquery** — since `EXISTS` only cares whether a
  row exists, not what's in it, the selected column is arbitrary.
  `SELECT 1` is the conventional way to signal "the columns don't matter
  here" — some people write `SELECT *` instead; both work identically.
- **Correlated subquery** — the line `np.person_id = p.person_id` is what
  makes this different from 6.1's subquery: it reaches *outside* the
  subquery to reference the outer query's current row (`p.person_id`).
  This means the subquery isn't computed once up front (like 6.1's `NOT
  IN` list was); conceptually, it re-runs for each outer row, asking "does
  a Directing nomination exist *for this specific person*?"
- **Why this is often preferred over `NOT IN`** — `NOT IN` has a sharp
  edge: if the subquery's result list contains even one `NULL`, `NOT IN`
  silently returns *zero rows for the entire outer query* — because
  comparing anything to `NULL` yields "unknown," not false, and SQL
  treats "unknown" as a non-match everywhere. `NOT EXISTS` only checks
  presence/absence, so it isn't affected by stray `NULL`s in the
  subquery's columns. It doesn't change today's result (`person_id` is
  never `NULL` here), but it's the reason experienced SQL writers often
  reach for `NOT EXISTS` by default rather than `NOT IN`.

### 7.2 Naming a subquery: Common Table Expressions

__Q: Same question as 6.4 — which Best Picture winners had no Directing
nomination and no Acting win? — with the subquery pulled out and named.__

6.4's `NOT IN` subquery works, but it's buried inline inside `WHERE`,
several lines deep, by the time a reader gets to it. A **CTE**
(`WITH ... AS`) lets you compute that same subquery first, give it a
plain-English name, and then just reference the name — turning "some
films are excluded for a two-part reason typed here in the middle of a
WHERE clause" into "here's a list called `excluded_films`, and here's
what disqualifies a film."

**Key syntax:** `WITH name AS (...)`, referencing a CTE like a table

**Docs:** [WITH clause / Common Table Expressions](https://www.sqlite.org/lang_with.html)

In [24]:
result = q("""
WITH excluded_films AS (
    SELECT nf2.film_id
    FROM nomination_films nf2
    JOIN nominations n2 ON nf2.nomination_id = n2.nomination_id
    JOIN categories cat2 ON n2.category_id = cat2.category_id
    WHERE cat2.award_group = 'Directing'
       OR (cat2.class = 'Acting' AND n2.is_winner = 1)
)
SELECT f.title, c.year_label
FROM films f
JOIN nomination_films nf ON f.film_id = nf.film_id
JOIN nominations n ON nf.nomination_id = n.nomination_id
JOIN categories cat ON n.category_id = cat.category_id
JOIN ceremonies c ON n.ceremony = c.ceremony
WHERE n.is_winner = 1
  AND cat.award_group = 'Best Picture'
  AND f.film_id NOT IN (SELECT film_id FROM excluded_films)
ORDER BY c.ceremony;
""")

result

,title,year_label
1,Wings,1927/28
2,Grand Hotel,1931/32
3,Argo,2012


__Notes__

- **`WITH name AS (...)`** — defines a temporary, named result set that
  exists only for the query it's attached to. It's computed as if it were
  a real table, then can be referenced by name anywhere later in the same
  query — here, `SELECT film_id FROM excluded_films` reads exactly like
  querying an actual table, even though `excluded_films` doesn't exist
  anywhere in the schema.
- **Identical result to 6.4** — this produces exactly the same three
  films. A CTE doesn't change what a query computes, only how the query
  reads. The exclusion logic (Directing nomination, or an Acting win) is
  unchanged; it's just been given a name and moved above the main query
  instead of sitting inline inside `WHERE`.
- **Why this is worth the extra syntax** — 6.4's version required
  reading through nested joins and an `OR`/`AND` condition *before*
  reaching the point where you could tell what the subquery was even for.
  Naming it `excluded_films` upfront tells the reader the point before
  they read a single line of its logic.
- **Not correlated, unlike 7.1** — this CTE has no reference back to the
  outer query (no `p.person_id`-style tie-back). It's computed once,
  independently, and reused — closer in spirit to 6.1's original `NOT IN`
  subquery than to 7.1's `NOT EXISTS`.
- **Multiple CTEs** — more than one can be defined in a single `WITH`,
  comma-separated (`WITH a AS (...), b AS (...)`), each able to reference
  earlier ones. Not needed here, but this is why CTEs scale well once a
  query needs several intermediate named sets rather than just one.

## Part 8 — Computed columns and value transformation

**What we're learning:** producing values that don't exist as-is in any
column — through arithmetic, fallback logic, or (in Round 2)
conditional expressions.


### 8.1 Arithmetic on columns, sorting by the result

__Q: Who are the youngest and oldest Oscar acting winners?__

Age at win isn't a stored fact — it's approximated here as
`film.release_year - person.birth_year`, since the database has no exact
birthdate or ceremony date. Scope is narrowed to acting categories,
since those are the only nominations tied to a specific performance in a
specific film, making "the film's release year" a meaningful stand-in for
"roughly when this happened" — though ceremonies are typically held the
year *after* a film's release, so treat this as approximate, not exact.

**Key syntax:** arithmetic directly in `SELECT`, `ORDER BY` on a computed
column, `IS NOT NULL` (from Part 2) protecting the arithmetic

**Docs:** [Expressions](https://www.sqlite.org/lang_expr.html)


In [18]:
result = q("""
SELECT p.name, p.birth_year, f.title, f.release_year,
       (f.release_year - p.birth_year) AS approx_age_at_win
FROM nominations n
JOIN categories cat ON n.category_id = cat.category_id
JOIN nomination_people np ON n.nomination_id = np.nomination_id
JOIN people p ON np.person_id = p.person_id
JOIN nomination_films nf ON n.nomination_id = nf.nomination_id
JOIN films f ON nf.film_id = f.film_id
WHERE n.is_winner = 1
  AND cat.class = 'Acting'
  AND p.birth_year IS NOT NULL     -- protects the arithmetic below (see Part 2.2)
ORDER BY approx_age_at_win ASC
LIMIT 5;
""")

result

,name,birth_year,title,release_year,approx_age_at_win
1,Tatum O'Neal,1963,Paper Moon,1973,10
2,Anna Paquin,1982,The Piano,1993,11
3,Patty Duke,1946,The Miracle Worker,1962,16
4,Timothy Hutton,1960,Ordinary People,1980,20
5,Janet Gaynor,1906,7th Heaven,1927,21


In [19]:
result = q("""
SELECT p.name, p.birth_year, f.title, f.release_year,
       (f.release_year - p.birth_year) AS approx_age_at_win
FROM nominations n
JOIN categories cat ON n.category_id = cat.category_id
JOIN nomination_people np ON n.nomination_id = np.nomination_id
JOIN people p ON np.person_id = p.person_id
JOIN nomination_films nf ON n.nomination_id = nf.nomination_id
JOIN films f ON nf.film_id = f.film_id
WHERE n.is_winner = 1
  AND cat.class = 'Acting'
  AND p.birth_year IS NOT NULL
ORDER BY approx_age_at_win DESC     -- only difference from the cell above: DESC instead of ASC
LIMIT 5;
""")

result

,name,birth_year,title,release_year,approx_age_at_win
1,Anthony Hopkins,1937,The Father,2020,83
2,Christopher Plummer,1929,Beginners,2010,81
3,Jessica Tandy,1909,Driving Miss Daisy,1989,80
4,George Burns,1896,The Sunshine Boys,1975,79
5,Melvyn Douglas,1901,Being There,1979,78


__Notes__

- **Arithmetic directly in `SELECT`** —
  `(f.release_year - p.birth_year)` computes a new value from two existing
  columns, on the fly, without needing that value stored anywhere.
  `AS approx_age_at_win` names the computed column, the same aliasing role
  as `AS n` / `AS wins` earlier.
- **`ORDER BY` on a computed column** — sorting isn't limited to columns
  that exist in a table; any `SELECT` expression, including a computed
  one, can be sorted on. `ASC` (default) surfaces the smallest values
  first (youngest); `DESC` reverses it (oldest) — the only difference
  between the two cells above.
- **`IS NOT NULL` protecting arithmetic** — excludes rows where
  `birth_year` is missing, since `NULL` arithmetic just silently produces
  another `NULL` rather than an error (see Part 2.2). Without this filter,
  people with no `birth_year` would appear in the result with a blank
  `approx_age_at_win`, sorting unpredictably.
- **No `Int64` cast needed here** — unlike 2.1, `IS NOT NULL` removes
  every row with a missing `birth_year` before the result is even built,
  so the returned column never contains a `NULL`/`NaN` in the first place,
  and pandas keeps it as a clean integer type without any float upcast.


### 8.2 Falling back to a default value

__Q: What are the Chinese titles for the 2019 Best Picture nominees, with
the English title shown wherever no Chinese title exists?__

**Key syntax:** `COALESCE`

**Docs:** [COALESCE](https://www.sqlite.org/lang_corefunc.html#coalesce)


In [20]:
result = q("""
SELECT f.title, f.title_zh, COALESCE(f.title_zh, f.title) AS display_title
FROM films f
JOIN nomination_films nf ON f.film_id = nf.film_id
JOIN nominations n ON nf.nomination_id = n.nomination_id
JOIN categories cat ON n.category_id = cat.category_id
JOIN ceremonies c ON n.ceremony = c.ceremony
WHERE cat.award_group = 'Best Picture'
  AND c.year_label = '2019'
ORDER BY f.title;
""")

result

,title,title_zh,display_title
1,1917,NaN,1917
2,Ford v Ferrari,极速车王,极速车王
3,Jojo Rabbit,乔乔的异想世界,乔乔的异想世界
4,Joker,小丑,小丑
5,Little Women,小妇人,小妇人
6,Marriage Story,婚姻故事,婚姻故事
7,Once Upon a Time in... Hollywood,好莱坞往事,好莱坞往事
8,Parasite,寄生虫,寄生虫
9,The Irishman,爱尔兰人,爱尔兰人


__Notes__

- **`COALESCE(a, b)`** — returns the first non-NULL value among its
  arguments, left to right. `COALESCE(title_zh, title)` reads as: "use
  `title_zh` if it exists, otherwise fall back to `title`." With more than
  two arguments, it keeps trying each in order until one is non-NULL.
- **Join chain for filtering, `COALESCE` for display** —
  `films → nomination_films → nominations → categories`, plus
  `ceremonies`, narrows the result to one category in one year (same
  pattern as 3.1 and 6.1). `COALESCE` itself only operates on the `films`
  columns; the joins just control *which* films appear.
- **NULL as an expected gap, not missing data** — `title_zh` is `NULL` for
  roughly three-quarters of films (see `data_notes.md`, Source 5): IMDb's
  `title.akas` has no unambiguous Chinese row for them. `COALESCE` handles
  both "confirmed no Chinese title" and "genuinely uses the English title"
  identically at display time, without needing to know which is which.
- **The fallback is never stored** — per `schema.md`'s display rule,
  `films.title_zh` is left `NULL` rather than filled with a copy of
  `title`. Storing the fallback would erase the distinction between
  "confirmed Chinese title" and "no Chinese title found"; `display_title`
  here is recomputed fresh on every query instead.


### 8.3 Conditional expressions

__Q: How many acting wins fall into each rough age bracket — under 30,
30s–40s, 50 and over?__

8.1 computed an exact `approx_age_at_win` for every acting winner.
Sometimes what's actually useful isn't the exact number but which *bucket*
it falls into — and that requires branching logic inside `SELECT`, which
is what `CASE WHEN` is for.

**Key syntax:** `CASE WHEN ... THEN ... ELSE ... END`

**Docs:** [CASE expression](https://www.sqlite.org/lang_expr.html#the_case_expression)

In [26]:
result = q("""
SELECT
    CASE
        WHEN (f.release_year - p.birth_year) < 30 THEN 'Under 30'
        WHEN (f.release_year - p.birth_year) < 50 THEN '30-49'
        ELSE '50 and over'
    END AS age_bracket,
    COUNT(*) AS wins
FROM nominations n
JOIN categories cat ON n.category_id = cat.category_id
JOIN nomination_people np ON n.nomination_id = np.nomination_id
JOIN people p ON np.person_id = p.person_id
JOIN nomination_films nf ON n.nomination_id = nf.nomination_id
JOIN films f ON nf.film_id = f.film_id
WHERE n.is_winner = 1
  AND cat.class = 'Acting'
  AND p.birth_year IS NOT NULL
GROUP BY age_bracket
ORDER BY
    CASE age_bracket                 -- a second CASE, just for sort order
        WHEN 'Under 30' THEN 1
        WHEN '30-49' THEN 2
        WHEN '50 and over' THEN 3
    END;
""")

result

,age_bracket,wins
1,Under 30,57
2,30-49,224
3,50 and over,100


__Notes__

- **`CASE WHEN ... THEN ... ELSE ... END`** — evaluates each `WHEN`
  condition in order, top to bottom, and returns the `THEN` value for the
  *first* one that's true. `ELSE` catches everything that didn't match any
  `WHEN` above it. The whole expression produces one value per row, just
  like `(f.release_year - p.birth_year)` did in 8.1 — it can be aliased
  (`AS age_bracket`) and used anywhere a normal column could be.
- **Order matters inside the bracket logic** — `WHEN (...) < 30` is
  checked first, so anyone under 30 is caught there and never reaches the
  `< 50` check. If the conditions were reversed (`< 50` first), everyone
  under 30 would incorrectly land in `'30-49'`, since they also satisfy
  `< 50` and it would be checked first. Whenever ranges are involved,
  order the `WHEN`s from narrowest/most specific to broadest.
- **`GROUP BY age_bracket`** — grouping by a `CASE WHEN` expression's
  *alias* works the same way as grouping by any computed column (see 8.1's
  `ORDER BY approx_age_at_win`). SQLite groups rows by the bracket they
  landed in, then `COUNT(*)` (Part 4) counts how many fell into each one.
- **A second `CASE WHEN`, used only for sorting** — `age_bracket` is text,
  so a plain `ORDER BY age_bracket` would sort it alphabetically
  (`'30-49'`, `'50 and over'`, `'Under 30'`) — not the actual age
  sequence. The `ORDER BY` clause maps each label to a small integer
  (1, 2, 3) reflecting the order the brackets should read in, and sorts on
  that number instead of the text. This is a common `CASE WHEN` use on its
  own: whenever a category has a natural order that doesn't match
  alphabetical order, a small lookup `CASE` inside `ORDER BY` fixes it.
- **Reuses `IS NOT NULL` and the join chain from 8.1** — same underlying
  query, same reason for excluding missing `birth_year`s (arithmetic
  against `NULL` silently produces `NULL`, which would form its own
  meaningless bracket otherwise); the new part is wrapping the arithmetic
  in one `CASE WHEN` for the bracket, and a second `CASE WHEN` purely to
  control the row order.